Tâche 1 — Comprendre BERT et XLM-RoBERTa

In [68]:
!pip install transformers torch scikit-learn --quiet

import zipfile

zip_path = "/content/Basics of BERT and XLM-RoBERTa - PyTorch - 2.zip"

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall("/content/data")
    print("Fichiers extraits :")
    print(z.namelist())

Fichiers extraits :
['Basics of BERT and XLM-RoBERTa - PyTorch/sample_submission.csv', 'Basics of BERT and XLM-RoBERTa - PyTorch/test.csv.zip', 'Basics of BERT and XLM-RoBERTa - PyTorch/train.csv.zip']


In [69]:
import pandas as pd
import numpy as np

train_df = pd.read_csv("/content/data/Basics_of_BERT_and_XLM-RoBERTa-PyTorch/train.csv.zip")
test_df  = pd.read_csv("/content/data/Basics_of_BERT_and_XLM-RoBERTa-PyTorch/test.csv.zip")

# Définir label_names ici pour toute la suite
label_names = {0: "Entailment", 1: "Neutral", 2: "Contradiction"}

print("=== Structure du dataset ===")
print(f"Train shape : {train_df.shape}")
print(f"Test shape  : {test_df.shape}")
print(f"\nColonnes : {train_df.columns.tolist()}")
print(train_df.head())

=== Structure du dataset ===
Train shape : (12120, 6)
Test shape  : (5195, 5)

Colonnes : ['id', 'premise', 'hypothesis', 'lang_abv', 'language', 'label']
           id                                            premise  \
0  5130fd2cb5  and these comments were considered in formulat...   
1  5b72532a0b  These are issues that we wrestle with in pract...   
2  3931fbe82a  Des petites choses comme celles-là font une di...   
3  5622f0c60b  you know they can't really defend themselves l...   
4  86aaa48b45  ในการเล่นบทบาทสมมุติก็เช่นกัน โอกาสที่จะได้แสด...   

                                          hypothesis lang_abv language  label  
0  The rules developed in the interim were put to...       en  English      0  
1  Practice groups are not permitted to work on t...       en  English      2  
2              J'essayais d'accomplir quelque chose.       fr   French      0  
3  They can't defend themselves because of their ...       en  English      0  
4    เด็กสามารถเห็นได้ว่าชาติพันธุ์แ

In [70]:
# Cellule 1 : Installation
!pip install transformers torch scikit-learn --quiet

In [71]:
# Cellule 2 : Charger les deux tokenizers et comparer
from transformers import BertTokenizer, XLMRobertaTokenizer

# Charger BERT
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Charger XLM-RoBERTa
xlm_tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')

In [72]:
phrase = "Drogba scored a beautiful goal in Abidjan"

# Comparer les tokenisations
bert_tokens = bert_tokenizer.tokenize(phrase)
xlm_tokens  = xlm_tokenizer.tokenize(phrase)

In [73]:
print(f"Phrase : {phrase}")
print(f"\nBERT tokens    : {bert_tokens}")
print(f"XLM-R tokens   : {xlm_tokens}")
print(f"\nTaille vocab BERT  : {bert_tokenizer.vocab_size}")
print(f"Taille vocab XLM-R : {xlm_tokenizer.vocab_size}")
print(f"\nTokens spéciaux BERT  : {bert_tokenizer.special_tokens_map}")
print(f"Tokens spéciaux XLM-R : {xlm_tokenizer.special_tokens_map}")

Phrase : Drogba scored a beautiful goal in Abidjan

BERT tokens    : ['dr', '##og', '##ba', 'scored', 'a', 'beautiful', 'goal', 'in', 'ab', '##id', '##jan']
XLM-R tokens   : ['▁Drog', 'ba', '▁score', 'd', '▁a', '▁beautiful', '▁goal', '▁in', '▁Abid', 'jan']

Taille vocab BERT  : 30522
Taille vocab XLM-R : 250002

Tokens spéciaux BERT  : {'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}
Tokens spéciaux XLM-R : {'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}


Tâche 2 — Tokenisation avec encode_plus

In [74]:
# Pour NLI on tokenise DEUX phrases ensemble : premise + hypothesis
from transformers import BertTokenizer, XLMRobertaTokenizer

bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
xlm_tokenizer  = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')

In [75]:
# Exemple du dataset — maintenant train_df existe
premise    = train_df['premise'].iloc[0]
hypothesis = train_df['hypothesis'].iloc[0]
label      = train_df['label'].iloc[0]

print(f"Premise    : {premise[:60]}...")
print(f"Hypothesis : {hypothesis[:60]}...")
print(f"Label      : {label} ({label_names[label]})")

# Tokeniser deux phrases ensemble
bert_encoded = bert_tokenizer(
    premise, hypothesis,
    max_length=128,
    padding='max_length',
    truncation=True,
    return_tensors='pt'
)

xlm_encoded = xlm_tokenizer(
    premise, hypothesis,
    max_length=128,
    padding='max_length',
    truncation=True,
    return_tensors='pt'
)

print(f"\n=== BERT ===")
print(f"input_ids shape      : {bert_encoded['input_ids'].shape}")
print(f"attention_mask shape : {bert_encoded['attention_mask'].shape}")
tokens = bert_tokenizer.convert_ids_to_tokens(bert_encoded['input_ids'][0])
print(f"Premiers tokens : {tokens[:10]}")
print(f"Derniers tokens : {tokens[-5:]}")

print(f"\n=== XLM-RoBERTa ===")
print(f"input_ids shape      : {xlm_encoded['input_ids'].shape}")
print(f"attention_mask shape : {xlm_encoded['attention_mask'].shape}")
tokens_xlm = xlm_tokenizer.convert_ids_to_tokens(xlm_encoded['input_ids'][0])
print(f"Premiers tokens : {tokens_xlm[:10]}")

print(f"\nDécodage BERT  : {bert_tokenizer.decode(bert_encoded['input_ids'][0])[:80]}...")
print(f"Décodage XLM-R : {xlm_tokenizer.decode(xlm_encoded['input_ids'][0])[:80]}...")

# Test sur deux phrases simples
phrase1 = "Drogba is a great player"
phrase2 = "He scored many goals"

bert_two   = bert_tokenizer(phrase1, phrase2, max_length=30, padding='max_length', truncation=True, return_tensors='pt')
tokens_two = bert_tokenizer.convert_ids_to_tokens(bert_two['input_ids'][0])
print(f"\nDeux phrases simples :")
print(f"Tokens : {tokens_two}")
print(f"\n→ [CLS] au début, [SEP] entre les deux phrases et à la fin")

Premise    : and these comments were considered in formulating the interi...
Hypothesis : The rules developed in the interim were put together with th...
Label      : 0 (Entailment)

=== BERT ===
input_ids shape      : torch.Size([1, 128])
attention_mask shape : torch.Size([1, 128])
Premiers tokens : ['[CLS]', 'and', 'these', 'comments', 'were', 'considered', 'in', 'formula', '##ting', 'the']
Derniers tokens : ['[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']

=== XLM-RoBERTa ===
input_ids shape      : torch.Size([1, 128])
attention_mask shape : torch.Size([1, 128])
Premiers tokens : ['<s>', '▁and', '▁these', '▁comments', '▁were', '▁considered', '▁in', '▁formula', 'ting', '▁the']

Décodage BERT  : [CLS] and these comments were considered in formulating the interim rules. [SEP]...
Décodage XLM-R : <s> and these comments were considered in formulating the interim rules.</s></s>...

Deux phrases simples :
Tokens : ['[CLS]', 'dr', '##og', '##ba', 'is', 'a', 'great', 'player', '[SEP]', 'he', 's

Tâche 3 — Préparation des données d'entrée

In [76]:
from transformers import BertTokenizer, XLMRobertaTokenizer

bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
xlm_tokenizer  = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')

phrase = "Drogba scored a beautiful goal in Abidjan"

bert_tokens = bert_tokenizer.tokenize(phrase)
xlm_tokens  = xlm_tokenizer.tokenize(phrase)

print(f"Phrase : {phrase}")
print(f"\nBERT tokens    : {bert_tokens}")
print(f"XLM-R tokens   : {xlm_tokens}")
print(f"\nTaille vocab BERT  : {bert_tokenizer.vocab_size}")
print(f"Taille vocab XLM-R : {xlm_tokenizer.vocab_size}")
print(f"\nTokens spéciaux BERT  : {bert_tokenizer.special_tokens_map}")
print(f"Tokens spéciaux XLM-R : {xlm_tokenizer.special_tokens_map}")

Phrase : Drogba scored a beautiful goal in Abidjan

BERT tokens    : ['dr', '##og', '##ba', 'scored', 'a', 'beautiful', 'goal', 'in', 'ab', '##id', '##jan']
XLM-R tokens   : ['▁Drog', 'ba', '▁score', 'd', '▁a', '▁beautiful', '▁goal', '▁in', '▁Abid', 'jan']

Taille vocab BERT  : 30522
Taille vocab XLM-R : 250002

Tokens spéciaux BERT  : {'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}
Tokens spéciaux XLM-R : {'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}


In [77]:
import torch

print("=== Tokens spéciaux BERT ===")
print(bert_tokenizer.special_tokens_map)
print(f"Vocab size : {bert_tokenizer.vocab_size}")

print("\n=== Tokens spéciaux XLM-RoBERTa ===")
print(xlm_tokenizer.special_tokens_map)
print(f"Vocab size : {xlm_tokenizer.vocab_size}")

# Fonction pour une seule phrase
def prepare_input(text, tokenizer, max_length=64):
    encoded = tokenizer(
        text,
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )
    return {
        'input_ids':      encoded['input_ids'],
        'attention_mask': encoded['attention_mask']
    }

# Fonction pour NLI (deux phrases)
def prepare_nli_input(premise, hypothesis, tokenizer, max_length=128):
    encoded = tokenizer(
        premise,
        hypothesis,
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )
    return {
        'input_ids':      encoded['input_ids'],
        'attention_mask': encoded['attention_mask']
    }

# Test phrase unique
test_phrase = "MTN Côte d'Ivoire offre les meilleurs services"
result_bert = prepare_input(test_phrase, bert_tokenizer)
result_xlm  = prepare_input(test_phrase, xlm_tokenizer)

print(f"\nPhrase unique : '{test_phrase}'")
print(f"BERT  input_ids shape : {result_bert['input_ids'].shape}")
print(f"XLM-R input_ids shape : {result_xlm['input_ids'].shape}")

# Test NLI
result_nli = prepare_nli_input(premise, hypothesis, xlm_tokenizer)
print(f"\nNLI (premise + hypothesis) :")
print(f"input_ids shape      : {result_nli['input_ids'].shape}")
print(f"attention_mask shape : {result_nli['attention_mask'].shape}")
print(f"\nAttention mask (20 premiers) : {result_nli['attention_mask'][0][:20].tolist()}")
print("1 = token réel | 0 = padding ignoré par le modèle")

=== Tokens spéciaux BERT ===
{'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}
Vocab size : 30522

=== Tokens spéciaux XLM-RoBERTa ===
{'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}
Vocab size : 250002

Phrase unique : 'MTN Côte d'Ivoire offre les meilleurs services'
BERT  input_ids shape : torch.Size([1, 64])
XLM-R input_ids shape : torch.Size([1, 64])

NLI (premise + hypothesis) :
input_ids shape      : torch.Size([1, 128])
attention_mask shape : torch.Size([1, 128])

Attention mask (20 premiers) : [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
1 = token réel | 0 = padding ignoré par le modèle


Tâche 4 — Charger le dataset

In [78]:
print("=== Distribution des labels ===")
counts = train_df['label'].value_counts().sort_index()
for label_id, count in counts.items():
    pct = count / len(train_df) * 100
    print(f"  {label_id} ({label_names[label_id]}) : {count} exemples ({pct:.1f}%)")

print("\n=== Distribution des langues ===")
print(train_df['language'].value_counts())

print("\n=== Valeurs manquantes ===")
print(train_df.isnull().sum())

=== Distribution des labels ===
  0 (Entailment) : 4176 exemples (34.5%)
  1 (Neutral) : 3880 exemples (32.0%)
  2 (Contradiction) : 4064 exemples (33.5%)

=== Distribution des langues ===
language
English       6870
Chinese        411
Arabic         401
French         390
Swahili        385
Urdu           381
Vietnamese     379
Russian        376
Hindi          374
Greek          372
Thai           371
Spanish        366
Turkish        351
German         351
Bulgarian      342
Name: count, dtype: int64

=== Valeurs manquantes ===
id            0
premise       0
hypothesis    0
lang_abv      0
language      0
label         0
dtype: int64


Tâche 5 — K-Fold Validation Croisée

In [79]:
from sklearn.model_selection import StratifiedKFold

N_FOLDS = 5

X_premise    = train_df['premise'].values
X_hypothesis = train_df['hypothesis'].values
X_lang       = train_df['language'].values
y            = train_df['label'].values

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

train_folds = []
val_folds   = []

print(f"Création de {N_FOLDS} folds stratifiés...\n")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_premise, y)):
    fold_train = {
        'premise':    X_premise[train_idx],
        'hypothesis': X_hypothesis[train_idx],
        'language':   X_lang[train_idx],
        'label':      y[train_idx]
    }
    fold_val = {
        'premise':    X_premise[val_idx],
        'hypothesis': X_hypothesis[val_idx],
        'language':   X_lang[val_idx],
        'label':      y[val_idx]
    }

    train_folds.append(fold_train)
    val_folds.append(fold_val)

    train_counts = np.bincount(y[train_idx])
    val_counts   = np.bincount(y[val_idx])

    print(f"Fold {fold+1} :")
    print(f"  Train : {len(train_idx)} exemples")
    print(f"  Val   : {len(val_idx)} exemples")
    print(f"  Train labels → 0:{train_counts[0]} 1:{train_counts[1]} 2:{train_counts[2]}")
    print(f"  Val   labels → 0:{val_counts[0]}  1:{val_counts[1]}  2:{val_counts[2]}")
    print()

print(f" {N_FOLDS} folds créés !")

Création de 5 folds stratifiés...

Fold 1 :
  Train : 9696 exemples
  Val   : 2424 exemples
  Train labels → 0:3340 1:3104 2:3252
  Val   labels → 0:836  1:776  2:812

Fold 2 :
  Train : 9696 exemples
  Val   : 2424 exemples
  Train labels → 0:3341 1:3104 2:3251
  Val   labels → 0:835  1:776  2:813

Fold 3 :
  Train : 9696 exemples
  Val   : 2424 exemples
  Train labels → 0:3341 1:3104 2:3251
  Val   labels → 0:835  1:776  2:813

Fold 4 :
  Train : 9696 exemples
  Val   : 2424 exemples
  Train labels → 0:3341 1:3104 2:3251
  Val   labels → 0:835  1:776  2:813

Fold 5 :
  Train : 9696 exemples
  Val   : 2424 exemples
  Train labels → 0:3341 1:3104 2:3251
  Val   labels → 0:835  1:776  2:813

 5 folds créés !
